# Monte Carlo Options Pricer — ResultsEvery figure and table below is produced from a fixed seed. Reference contract:`S0 = K = 100`, `r = 5%`, `sigma = 20%`, `T = 1`.

In [ ]:
import sysfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent / "src"))import numpy as npimport pandas as pdfrom mcpricer.binomial import binomial_american_put_pricefrom mcpricer.black_scholes import bs_call_price, bs_put_pricefrom mcpricer.pricing import lsm_american_pricefrom mcpricer.reporting import (    greeks_table,    plot_convergence,    plot_variance_reduction,    plot_volatility_smiles,)BASE = dict(S0=100.0, K=100.0, r=0.05, sigma=0.2, T=1.0)FIGURES = Path.cwd().parent / "figures"FIGURES.mkdir(exist_ok=True)bs_call_price(**BASE)

## 1. ConvergenceThe Monte Carlo price with its 95% interval against the closed-form price, andthe decay of the standard error against the `N^{-1/2}` reference line.

In [ ]:
study = plot_convergence(FIGURES / "convergence.png")pd.DataFrame(study).set_index("n_paths")

## 2. Variance reductionBoth estimators are run at a matched path count, so the comparison is like forlike on payoff evaluations.

In [ ]:
rows = plot_variance_reduction(FIGURES / "variance_reduction.png")pd.DataFrame(rows)

In [ ]:
# Paths naive sampling would need to match the control variate's accuracy.naive, control = rows[0], rows[2]500_000 * (naive["standard_error"] / control["standard_error"]) ** 2

## 3. GreeksEach estimator against its closed-form value, with the discrepancy expressed instandard errors rather than absolute terms.

In [ ]:
pd.DataFrame(greeks_table())

## 4. American putLongstaff-Schwartz against a binomial tree, plus the early exercise premium overthe European price.

In [ ]:
lsm = lsm_american_price(**BASE, n_paths=500_000, n_steps=50, seed=5)pd.DataFrame(    [        {"method": "Longstaff-Schwartz", "price": lsm.price, "standard_error": lsm.standard_error},        {"method": "binomial tree", "price": binomial_american_put_price(**BASE, n_steps=4000), "standard_error": np.nan},        {"method": "European (Black-Scholes)", "price": bs_put_price(**BASE), "standard_error": np.nan},    ])

## 5. Volatility smilesGBM is flat by construction. Merton's downward jumps lift the low strikes;Heston's negative correlation between price and variance produces the equityskew.

In [ ]:
smiles = plot_volatility_smiles(FIGURES / "volatility_smiles.png")pd.DataFrame(    {k: v for k, v in smiles.items() if k != "strikes"},    index=pd.Index(smiles["strikes"], name="strike"),)